In [38]:
!pip install transformers accelerate pillow requests gradio
!pip install groq -q

In [39]:
from PIL import Image
import json
import os
import time
import base64
import re
from difflib import SequenceMatcher
from groq import Groq


In [53]:
# Colab with Google Drive
from google.colab import drive
drive.mount('/content/drive')

# API Configuration
GROQ_API_KEY = "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"  # Replace with your Groq API key

# Paths
IMAGE_DIR = "/content/drive/MyDrive/book_ocr_images"
OUTPUT_FILE = "/content/drive/MyDrive/book_ocr_results.json"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [61]:
print("Setting up Groq API client...")
client = Groq(api_key=GROQ_API_KEY)


Setting up Groq API client...


In [62]:
with open(os.path.join(IMAGE_DIR, "metadata.json"), "r", encoding="utf-8") as f:
    metadata = json.load(f)

print(f"Loaded {len(metadata)} samples")

Loaded 17 samples


In [68]:
def encode_image(image_path):
    """Encode image to base64"""
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode('utf-8')


def call_vision_api(image_path, prompt, model=MODEL_NAME, max_retries=3):
    """Call Groq VL model via API"""

    # Read and encode image
    image_base64 = encode_image(image_path)

    # Retry loop
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}},
                            {"type": "text", "text": prompt}
                        ]
                    }
                ],
                max_tokens=2000
            )
            return response.choices[0].message.content

        except Exception as e:
            error_msg = str(e)
            if "429" in error_msg or "rate" in error_msg.lower():
                wait_time = (attempt + 1) * 30
                print(f"    Rate limited, waiting {wait_time}s...")
                time.sleep(wait_time)
            else:
                return f"Error: {error_msg}"

    return "Error: Max retries exceeded"

def word_accuracy_wer(prediction, ground_truth):
    """Word Accuracy (WER-based)"""
    pred_words = set(prediction.lower().split())
    gt_words = set(ground_truth.lower().split())
    if len(gt_words) == 0:
        return 0
    return 1-len(pred_words & gt_words) / len(gt_words)


def character_accuracy_cer(prediction, ground_truth):
    """CER - 忽略格式差异"""
    # 预处理：去除markdown符号、空格差异
    pred = re.sub(r'[*#\s]+', '', prediction.lower())
    gt = re.sub(r'[*#\s]+', '', ground_truth.lower())

    matcher = SequenceMatcher(None, gt, pred)
    return 1 - matcher.ratio()  # 错误率

def paragraph_analysis(prediction, ground_truth):
    """Paragraph Structure Analysis - 改为错误率"""
    pred_paragraphs = [p.strip() for p in re.split(r'\n+', prediction) if p.strip()]
    gt_paragraphs = [p.strip() for p in re.split(r'\n+', ground_truth) if p.strip()]

    # 段落数量错误率
    pred_count = len(pred_paragraphs)
    gt_count = len(gt_paragraphs)
    count_error = abs(pred_count - gt_count) / max(gt_count, 1)

    # 平均段落长度错误率
    pred_avg_len = sum(len(p) for p in pred_paragraphs) / max(pred_count, 1)
    gt_avg_len = sum(len(p) for p in gt_paragraphs) / max(gt_count, 1)
    length_error = abs(pred_avg_len - gt_avg_len) / max(gt_avg_len, 1) if gt_avg_len > 0 else 0

    return {
        "pred_paragraphs": pred_count,
        "gt_paragraphs": gt_count,
        "count_error_rate": round(count_error, 2),      # 改成错误率
        "avg_length_error_rate": round(length_error, 2), # 改成错误率
        "structure_warning": count_error > 0.5 or length_error > 0.5
    }
def text_density_analysis(prediction, ground_truth):
    """Text Density Analysis - 改为错误率"""
    pred_words = prediction.lower().split()
    gt_words = ground_truth.lower().split()

    pred_word_count = len(pred_words)
    gt_word_count = len(gt_words)

    # 词数错误率
    word_count_error = abs(pred_word_count - gt_word_count) / max(gt_word_count, 1)

    # 密度错误率
    pred_density = len(prediction) / max(pred_word_count, 1)
    gt_density = len(ground_truth) / max(gt_word_count, 1)
    density_error = abs(pred_density - gt_density) / max(gt_density, 1) if gt_density > 0 else 0

    return {
        "pred_word_count": pred_word_count,
        "gt_word_count": gt_word_count,
        "word_count_error_rate": round(word_count_error, 2),  # 改成错误率
        "pred_density": round(pred_density, 2),
        "gt_density": round(gt_density, 2),
        "density_error_rate": round(density_error, 2),         # 改成错误率
        "density_warning": word_count_error > 0.5 or density_error > 0.5
    }

In [69]:
EXTRACTION_PROMPT = """Extract ALL visible text from this book page. Return only the raw text content, no descriptions, no summaries. Include all paragraphs."""
MODEL_NAME = "meta-llama/llama-4-scout-17b-16e-instruct"  # Groq vision model


In [73]:
results = []
total_word_acc = 0
total_char_acc = 0

for item in metadata:
    image_path = os.path.join(IMAGE_DIR, item["filename"])
    ground_truth = item["ground_truth"]

    # Call API instead of local model
    prediction = call_vision_api(image_path, EXTRACTION_PROMPT)

    # Skip scoring if error
    if prediction.startswith("Error:"):
        print(f"✗ {item['filename']}: {prediction}")
        results.append({
            "filename": item["filename"],
            "error": prediction,
            "ground_truth": ground_truth,
            "prediction": None
        })
        continue

    # Scoring
    word_acc = word_accuracy_wer(prediction, ground_truth)
    char_acc = character_accuracy_cer(prediction, ground_truth)
    para_analysis = paragraph_analysis(prediction, ground_truth)
    density_analysis = text_density_analysis(prediction, ground_truth)

    # Accumulate
    total_word_acc += word_acc
    total_char_acc += char_acc

    # Warning flags
    warning_flags = []
    if para_analysis["structure_warning"]:
        warning_flags.append("LAYOUT")
    if density_analysis["density_warning"]:
        warning_flags.append("DENSITY")

    warning_str = f" [{', '.join(warning_flags)}]" if warning_flags else ""

    results.append({
        "filename": item["filename"],
        "ground_truth": ground_truth,  # 完整保存
        "prediction": prediction,       # 完整保存
        "word_error_rate": round(word_acc, 4),  # 已改为错误率
        "char_error_rate": round(char_acc, 4),  # 已改为错误率
        "paragraph_analysis": para_analysis,
        "density_analysis": density_analysis,
        "layout_warning": para_analysis["structure_warning"],
        "density_warning": density_analysis["density_warning"]
    })

    print(f"✓ {item['filename']}: WER={word_acc:.2f}, CER={char_acc:.2f}{warning_str}")

✓ book_00.jpg: WER=0.18, CER=0.14
✓ book_03.jpg: WER=0.04, CER=0.04
✓ book_04.jpg: WER=0.07, CER=0.03
✓ book_05.jpg: WER=0.06, CER=0.03
✓ book_06.jpg: WER=0.06, CER=0.06
✓ book_07.jpg: WER=0.05, CER=0.04
✓ book_08.jpg: WER=0.06, CER=0.03
✓ book_09.jpg: WER=0.16, CER=0.45
✓ book_10.jpg: WER=0.17, CER=0.20 [LAYOUT]
✓ book_11.jpg: WER=0.11, CER=0.11
✓ book_12.jpg: WER=0.04, CER=0.25
✓ book_13.jpg: WER=0.09, CER=0.04
✓ book_14.jpg: WER=0.04, CER=0.31
✓ book_15.jpg: WER=0.07, CER=0.50
✓ book_16.jpg: WER=0.07, CER=0.04
✓ book_17.jpg: WER=0.05, CER=0.09 [LAYOUT]
✓ book_18.jpg: WER=0.11, CER=0.25 [LAYOUT]


In [74]:
total = len(results)
successful = sum(1 for r in results if "error" not in r)
avg_word_acc = total_word_acc / successful * 100 if successful > 0 else 0
avg_char_acc = total_char_acc / successful * 100 if successful > 0 else 0

print(f"Exact Match Accuracy: {avg_word_acc:.2f}%")
print(f"Contains Match Accuracy: {avg_char_acc:.2f}%")

Exact Match Accuracy: 8.34%
Contains Match Accuracy: 15.21%


In [75]:
output_data = {
    "model": MODEL_NAME,
    "task": "Book OCR - Text Extraction via API",
    "total_samples": total,
    "successful": successful,
    "average_word_accuracy": f"{avg_word_acc:.2f}%",
    "average_char_accuracy": f"{avg_char_acc:.2f}%",
    "results": results
}

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"\n{'='*60}")
print(f"Model: {MODEL_NAME}")
print(f"Task: Book OCR - Text Extraction via API")
print(f"Total: {total}, Successful: {successful}")
print(f"Average Word Accuracy: {avg_word_acc:.2f}%")
print(f"Average Char Accuracy: {avg_char_acc:.2f}%")
print(f"\nResults saved to: {OUTPUT_FILE}")



Model: meta-llama/llama-4-scout-17b-16e-instruct
Task: Book OCR - Text Extraction via API
Total: 17, Successful: 17
Average Word Accuracy: 8.34%
Average Char Accuracy: 15.21%

Results saved to: /content/drive/MyDrive/book_ocr_results.json
